In [1]:
# spark sesson setup
import math
import h3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, to_timestamp, year, month, dayofweek, hour, expr, count, avg, round
from pyspark.sql.types import StringType, DoubleType
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("NYC_Taxi_H3_Processing") \
    .config("spark.driver.memory", "2g") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .getOrCreate()

print("Spark Session Created Successfully!")

2026-09-03 07:23:25,532 WARN util.Utils: Your hostname, localhost.localdomain resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
2026-09-03 07:23:25,535 WARN util.Utils: Set SPARK_LOCAL_IP if you need to bind to another address
2026-09-03 07:23:27,556 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark Session Created Successfully!


In [2]:
# H3 geohashing udf
@udf(returnType=StringType())
def lat_lng_to_h3(lat, lng):
    if lat is None or lng is None:
        return None
    try:
        return h3.latlng_to_cell(float(lat), float(lng), 8)
    except:
        return None

In [3]:
# haversine distance udf
@udf(returnType=DoubleType())
def haversine_distance(lat1, lon1, lat2, lon2):
    if None in (lat1, lon1, lat2, lon2):
        return None
    try:
        r = 3958.8  # radius in miles
        phi1, phi2 = math.radians(float(lat1)), math.radians(float(lat2))
        dphi = math.radians(float(lat2) - float(lat1))
        dlambda = math.radians(float(lon2) - float(lon1))
        a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
        return r * (2 * math.atan2(math.sqrt(a), math.sqrt(1 - a)))
    except:
        return None

In [4]:
df_bronze = spark.read.parquet("hdfs://localhost:9000/data/raw/rides/*.parquet")
print(f"Bronze Count: {df_bronze.count()}")

Bronze Count: 13380122


In [5]:
# 1. Formatting
df_formatted = df_bronze.withColumn("pickup_datetime", to_timestamp(col("Trip_Pickup_DateTime")))

# 2. Filtering invalid coordinates & zero values
df_clean_filtered = df_formatted.filter(
    (col("Start_Lat").isNotNull()) & (col("Start_Lon").isNotNull()) &
    (col("End_Lat").isNotNull()) & (col("End_Lon").isNotNull()) &
    (col("Start_Lat") != 0.0) & (col("Start_Lon") != 0.0) &
    (col("Passenger_Count") > 0) & (col("Trip_Distance") > 0)
)

# 3. Taking a stable 5% sample (~650k records) 
df_clean = df_clean_filtered.sample(withReplacement=False, fraction=0.05, seed=42).repartition(10)
print(f"Clean Sample Count: {df_clean.count()}")

Clean Sample Count: 654008


In [6]:
df_silver = df_clean \
    .withColumn("h3_index", lat_lng_to_h3(col("Start_Lat"), col("Start_Lon"))) \
    .withColumn("haversine_miles", round(haversine_distance(col("Start_Lat"), col("Start_Lon"), col("End_Lat"), col("End_Lon")), 2)) \
    .withColumn("pickup_year", year(col("pickup_datetime"))) \
    .withColumn("pickup_month", month(col("pickup_datetime"))) \
    .withColumn("pickup_hour", hour(col("pickup_datetime"))) \
    .withColumn("day_of_week", dayofweek(col("pickup_datetime"))) \
    .withColumn("is_weekend", expr("CASE WHEN day_of_week IN (1, 7) THEN 1 ELSE 0 END"))

df_silver.write \
    .mode("overwrite") \
    .partitionBy("pickup_year", "pickup_month") \
    .option("compression", "snappy") \
    .parquet("hdfs://localhost:9000/data/staging/rides_geo/")

print("Silver Layer Saved Successfully to HDFS!")

2026-09-03 07:24:40,655 WARN util.package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Silver Layer Saved Successfully to HDFS!


In [7]:
df_gold_hotspots = df_silver.groupBy("h3_index", "pickup_hour") \
    .agg(
        count("*").alias("total_trips"),
        round(avg("Fare_Amt"), 2).alias("avg_fare"),
        round(avg("Trip_Distance"), 2).alias("avg_recorded_distance"),
        round(avg("haversine_miles"), 2).alias("avg_haversine_distance")
    )

window_spec = Window.partitionBy("h3_index").orderBy("pickup_hour")

df_gold_final = df_gold_hotspots.withColumn(
    "running_avg_fare",
    round(avg("avg_fare").over(window_spec), 2)
)

df_gold_final.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet("hdfs://localhost:9000/data/gold/rides_summary/")

print("Gold Layer Saved Successfully to HDFS!")
df_gold_final.show(10, False)

Gold Layer Saved Successfully to HDFS!


+---------------+-----------+-----------+--------+---------------------+----------------------+----------------+
|h3_index       |pickup_hour|total_trips|avg_fare|avg_recorded_distance|avg_haversine_distance|running_avg_fare|
+---------------+-----------+-----------+--------+---------------------+----------------------+----------------+
|882a10052dfffff|13         |1          |5.7     |0.5                  |0.04                  |5.7             |
|882a1005b5fffff|11         |1          |7.7     |1.84                 |1.64                  |7.7             |
|882a100841fffff|12         |1          |6.1     |0.94                 |0.49                  |6.1             |
|882a100d23fffff|0          |1006       |9.26    |2.5                  |1.85                  |9.26            |
|882a100d23fffff|1          |797        |9.37    |2.68                 |2.03                  |9.32            |
|882a100d23fffff|2          |613        |9.99    |2.98                 |2.24                  |9